# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarisBinHabib/Internship-Repo/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*
## 1. My lane (or freestyle) and why.

I want to select Lane 2: Refresh / Content Opportunity Scoring.

I have already used the starter notebooks (01 and 02) and this workflow is in the same order end to end:
When the rule based baseline scored pages are transparent, a trained model prevailed clearly
(baseline precision@50 = 0.240 vs. random forest precision@50 = 0.740). That gap tells me
there's real signal in this data in the form of a simple rule that is missing — which is exactly the kind of
problem that is important for a 7-week time period. I also like that this lane produces a concrete, useful
output: a sorted list of pages that an editor can modify directly, along with reason codes that give reasons.
For each page, explain why it is on the list. In comparison to signal analysis (Lane 1) or clustering (Lane 3),
This lane is more descriptive and makes me make a decision, take an action, and make a choice.
from day one — metric that fits the way the problem definition in the Skill is framed
written.

In [19]:
import os
os.chdir("/content")
if not os.path.exists("/content/repo"):
    !git clone https://github.com/HarisBinHabib/Internship-Repo.git /content/repo
%cd /content/repo

/content/repo


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.shape


(30000, 44)

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*
## 2. The question: decision, action, cost of a wrong call

This is an improvement for an editor to make, in which pages, from a large content inventory, should they make their improvement?
Review first – refresh, expand, prune or monitor – with limited review capacity.

Who will implement it: A content editor or a SEO strategist that has a set number of weekly hours to work (e.g. can edit 100 articles a week).
Only read ~50 pages each week). Currently they have no ranked starting point and would
otherwise choose pages at random, or on a hunch.

The action performed: For each page in the ranked queue, the editor either refreshes the page, or prevents it from being refreshed.
depending on the setting, expands, prunes, protects as is or flags for monitoring.
It has reason codes that are attached to its score.

The consequences of a bad recommendation:
A false positive refers to a page that is marked as being a problem by the system, but is not actually a problem.
  Utilizes an editor's precious review time on a page which didn't require it.
A wrongly answered no (a page that was actually declining, or under-capturing)
  Only sees the real opportunity or real problem until it becomes worse and more costly (clicks):
  Traffic/visibility over time.

Because editor's time is precious and finite, precision in the top of the ranked list is important
More than capturing all possible cases — hence the need for precision@K (for example, precision@50) is the
It is not accuracy overall, it's accuracy here.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Total pages in starter dataset: {len(df)}")
print(f"If an editor can review ~50 pages/week, that's {len(df)/50:.0f} weeks to review everything once.")


Total pages in starter dataset: 30000
If an editor can review ~50 pages/week, that's 600 weeks to review everything once.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*
## 3. Quick look at the data (2-3 real numbers)

Using the starter dataset (`content_refresh_anonymized.csv`, 30,000 rows), I pulled three
numbers that support spending 7 weeks on this lane:

1. How many pages currently show a declining trend, out of the total with enough traffic to
   matter.
2. How many pages match the "stale but still visible" pattern — meaning they still get
   traffic but haven't been touched in 6+ months, which is a realistic candidate pool for an
   editor's review queue.
3. How much a learned model beats a simple rule at this exact task, using the verified
   starter pipeline results (precision@50 comparison).

These numbers show two things: there's a large enough candidate pool to make ranking useful
(not too few pages), and a simple hand-written rule clearly leaves accuracy on the table that
a model can recover.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. How many pages are declining, among those with real traffic (avoid the avg_position=0
#    "no data" trap and low-volume noise)
visible = df[df["impressions_90d"] >= 100]
declining = visible[visible["trend_direction"] == "down"]
print(f"Visible pages (>=100 impressions/90d): {len(visible)}")
print(f"Of those, declining: {len(declining)} ({len(declining)/len(visible)*100:.1f}%)")

# 2. Stale-but-visible pages (a concrete candidate pool for review), matching the
#    starter guide's 'stale_visible_page' reason code definition
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(f"\nStale but still visible pages: {len(stale_visible)}")

# 3. Baseline vs learned model, from the verified starter pipeline results
#    (outputs/model_report.md / outputs/model_results.json)
print("\nPrecision@50 (top 50 pages reviewed first):")
print("  Baseline rule score: 0.240  (~12 of top 50 correct)")
print("  Random forest model: 0.740  (~37 of top 50 correct)")


Visible pages (>=100 impressions/90d): 22006
Of those, declining: 13152 (59.8%)

Stale but still visible pages: 17

Precision@50 (top 50 pages reviewed first):
  Baseline rule score: 0.240  (~12 of top 50 correct)
  Random forest model: 0.740  (~37 of top 50 correct)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*
## 4. Careful words: what I can and can't claim

**What this work CAN say:**
- We observed that a rule-based baseline and a trained model differ in how well they rank
  pages by review priority, measured on held-out data (client-holdout validation).
- We can say a page's ranking is "associated with" certain measurable signals — like
  staleness, trend direction, position, and impressions — not that any single signal *causes*
  a decline or an opportunity.
- The output is decision-support: a ranked list an editor can use to prioritize limited review
  time, with reason codes they can inspect and override.
- Results are directional and observational — based on what happened in the data, not on a
  controlled experiment.

**What this work will NEVER claim:**
- That refreshing a page *caused* it to recover — that requires a real experiment (e.g. A/B
  testing before/after a refresh), which this data alone cannot provide.
- That any result reveals a Google ranking factor or how a search algorithm works.
- That a score predicts what will happen with certainty — it estimates relative priority
  among candidates, not a guaranteed outcome.
- That the model is "correct" in some absolute sense — only that it beats a transparent
  baseline on a specific metric (precision@50), on this specific slice of data.

I'll use words like "observed," "associated with," "suggests," and "ranked candidate for
review" — and avoid words like "proves," "causes," or "guarantees."

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No code needed here — this section is a written claims policy, not a numeric check.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.